<a href="https://colab.research.google.com/github/mg320-dotcom/daejeon_swimming_pool/blob/main/%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%82%AC%EC%9D%B4%EC%96%B8%EC%8A%A4_%EC%B2%9C%EB%AC%B8%ED%95%99_%EA%B3%BC%EC%A0%9C2_(%EC%A7%80%EB%8F%84_%EA%B7%B8%EB%A6%AC%EA%B8%B0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import drive
drive.mount('/content/gdrive/')
dir1 = "/content/gdrive/MyDrive/python2026/"  #dir=directory

import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
import csv
import seaborn as sns
import folium

Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).


In [26]:
!pip install folium
import folium as g

## 대전시 소재 수영장 위치 및 운영 여부 확인 지도

In [27]:
#대전 수영장 자료 불러오기 대전광역시 민간체육시설 현황 (공공데이터포털)
daejeon_pool_df=pd.read_csv(dir1+'daejeon_swimming_pool.csv')
daejeon_pool_df.head()

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.12/di

,등록/신고,업종,업소명,소재지,면적,주관부서,요일구분,cleaning time start1,cleaning time end1,cleaning time start2,cleaning time end2,cleaning time start3,cleaning time end3,cleaning time start4,cleaning time end4,cleaning time start5,cleaning time end5
0,신고,종합체육시설업,사이언스도룡스포츠센터,대전광역시 유성구 대덕대로 581 (도룡동),"1,949",유성구청 문화관광과,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,신고,수영장업(실내),에프스포렉스,"대전 동구 우암로 128, 지하2층(삼성동)","11,711",동구청 관광문화체육과,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,신고,수영장업(실내),용운국제수영장,대전 동구 동부로 138 (용운동),"8,137",동구청 관광문화체육과,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,신고,수영장업(실내),스위밍키즈,"대전 동구 계족로 10, 지하1층(효동)",381,동구청 관광문화체육과,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,신고,수영장업(실내),한밭수영장,대전시 중구 대종로 373(부사동),"3,754",중구청 문화체육과,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [39]:
!pip install -U googlemaps
import pandas as pd
import requests
import os
from google.colab import userdata
import requests


# 코랩 금고에서 키를 가져옵니다.
KAKAO_API_KEY = userdata.get('KAKAO_API_KEY')

if KAKAO_API_KEY == 'YOUR_KEY_HERE':
    print("⚠️ 경고: API 키가 설정되지 않았습니다. 환경 변수를 확인하세요.")


def get_lat_lng_kakao(address):
    if pd.isna(address): return None, None

    url = 'https://dapi.kakao.com/v2/local/search/address.json'
    # 중요: 'KakaoAK ' 뒤에 한 칸 띄우고 키가 들어가야 합니다.
    headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}
    params = {'query': address}

    try:
        response = requests.get(url, headers=headers, params=params)
        address_data = response.json()

        # 에러 메시지가 왔는지 확인하기 위한 출력 (문제 해결용)
        if 'errorType' in address_data:
            print(f"API 에러 발생: {address_data['message']}")
            return None, None

        if address_data.get('documents'):
            result = address_data['documents'][0]
            return result['y'], result['x']
        else:
            return None, None
    except Exception as e:
        return None, None

# 2. 파일 불러오기
df = pd.read_csv(dir1+'daejeon_swimming_pool.csv')


print("변환을 시작합니다...")

# 3. '소재지' 컬럼을 사용하여 변환
df[['위도', '경도']] = df['소재지'].apply(lambda x: pd.Series(get_lat_lng_kakao(x)))

# 4. 결과 저장 (한글 깨짐 방지)
df.to_csv('daejeon_pool_with_coords.csv', index=False, encoding='utf-8-sig')

print("작업 완료! 'daejeon_pool_with_coords.csv' 파일을 확인하세요.")

변환을 시작합니다...
작업 완료! 'daejeon_pool_with_coords.csv' 파일을 확인하세요.


In [36]:
# 1위경도가 포함된 데이터 불러오기
df = pd.read_csv('daejeon_pool_with_coords.csv')

# 2. 결측치(위경도 없는 데이터) 제거
df = df.dropna(subset=['위도', '경도'])

# 3. 지도 시작점 설정 (대전 중심 좌표)
# 위도 36.35, 경도 127.38 정도가 대전 시청 근처입니다.
m = folium.Map(location=[36.3504, 127.3845], zoom_start=12)

# 4. 데이터프레임을 순회하며 마커 찍기
for _, row in df.iterrows():
    # 마커 클릭 시 보여줄 팝업 내용 (업소명 + 소재지)
    popup_text = f"<b>{row['업소명']}</b><br>{row['소재지']}"

    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row['업소명'],  # 마우스를 올렸을 때 뜨는 이름
        icon=folium.Icon(color='orange', icon='info-sign')
    ).add_to(m)

m

내가 자주 다니는 수영장의 입장 가능 시간 바로 확인하기

In [40]:
import pandas as pd
import folium
from datetime import datetime
import pytz  # 타임존 처리를 위한 라이브러리
import webbrowser
import os

# 1. 파일 불러오기
try:
    df = pd.read_csv('daejeon_pool_with_coords.csv')
except:
    print("CSV 파일을 찾을 수 없습니다.")

# ---------------------------------------------------------
# [수정] 한국 시간(KST)을 기준으로 현재 시간을 가져옵니다.
local_tz = pytz.timezone('Asia/Seoul')
input_dt = datetime.now(local_tz)
current_time_str = input_dt.strftime('%Y-%m-%d %H:%M:%S')
# ---------------------------------------------------------

def get_detailed_status(row, input_dt):
    col_name = '요일구분 ' if '요일구분 ' in row else '요일구분'
    if pd.isna(row[col_name]): return None, None, None

    weekday = input_dt.weekday()
    day_type = '평일' if weekday < 5 else ('토요일' if weekday == 5 else '일요일')
    if str(row[col_name]).strip() != day_type: return None, None, None

    check_times = []
    for i in range(1, 6):
        s, e = row.get(f'cleaning time start{i}'), row.get(f'cleaning time end{i}')
        if pd.notna(s) and pd.notna(e) and str(s).strip() != "":
            check_times.append((str(s).strip(), str(e).strip()))

    if not check_times: return 'orange', '정보 누락', '정비 시간 정보가 없습니다.'

    current_time = input_dt.time()
    next_break_start = None

    for s_str, e_str in check_times:
        try:
            start = datetime.strptime(s_str, '%H:%M').time()
            end = datetime.strptime(e_str, '%H:%M').time()

            if start <= current_time < end:
                # 정비 종료 시간 계산 (오늘 날짜와 결합)
                end_dt = local_tz.localize(datetime.combine(input_dt.date(), end))
                wait_min = int((end_dt - input_dt).total_seconds() / 60)
                return 'red', '입장 불가 (정비 중)', f"종료까지 {wait_min}분 남음"

            if current_time < start:
                if next_break_start is None or start < next_break_start:
                    next_break_start = start
        except: continue

    if next_break_start:
        start_dt = local_tz.localize(datetime.combine(input_dt.date(), next_break_start))
        left_min = int((start_dt - input_dt).total_seconds() / 60)
        return 'blue', '입장 가능', f"다음 정비까지 {left_min}분 남음"
    return 'blue', '입장 가능', "오늘 정비 종료"

# 지도 초기화
m = folium.Map(location=[36.372389, 127.323040], zoom_start=12.3)

# 마커 추가
for _, row in df.iterrows():
    if pd.isna(row['위도']) or pd.isna(row['경도']): continue
    color, status, msg = get_detailed_status(row, input_dt)
    if color is None: continue

    popup_text = f"""
    <div style='width:200px; font-family: sans-serif;'>
        <b style='font-size:14px;'>{row['업소명']}</b><br>
        <hr style='margin:5px 0;'>
        상태: <b style='color:{color}'>{status}</b><br>
        안내: {msg}<br>
        <small style='color:gray'>(한국시간 기준: {input_dt.strftime('%H:%M')})</small>
    </div>
    """

    folium.Marker(
        location=[row['위도'], row['경도']],
        popup=folium.Popup(popup_text, max_width=250),
        icon=folium.Icon(color=color, icon='info-sign')
    ).add_to(m)

# HTML 저장 및 확인
m.save("index.html") # GitHub Pages용으로 이름을 index.html로 저장하면 편리합니다.
print(f"현재 한국 시간({current_time_str}) 기준으로 업데이트 완료!")
m

현재 한국 시간(2026-03-20 23:38:32) 기준으로 업데이트 완료!


모바일 환경에서 확인하기